In [ ]:
import base64
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI


# -----------------------------
# Configuration
# -----------------------------
API_KEY = "..."  # replace with your OpenAI API key

MODELS = [
    "gpt-5.4-mini",
]

INPUT_PARQUET = f"input.parquet"
OUTPUT_CSV = f"output.parquet"

IMAGE_ROOT = Path("/Volumes/Extreme SSD/interactions/inat_images")

client = OpenAI(api_key=API_KEY)


# -----------------------------
# Helpers
# -----------------------------
def image_to_data_url(image_path: str) -> str:
    path = Path(image_path)

    if not path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime_map = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }

    mime_type = mime_map.get(path.suffix.lower())

    if mime_type is None:
        raise ValueError(f"Unsupported image type: {path.suffix}")

    image_bytes = path.read_bytes()
    encoded = base64.b64encode(image_bytes).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"


def ask_gpt(model_name: str, image_path: str, caption: str) -> str:
    image_data_url = image_to_data_url(image_path)

    response = client.responses.create(
        model=model_name,
        temperature=0.0,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": caption},
                    {"type": "input_image", "image_url": image_data_url},
                ],
            }
        ],
    )

    return response.output_text.strip()


# -----------------------------
# Main evaluation
# -----------------------------
def main() -> None:
    df = pd.read_parquet(
        INPUT_PARQUET,
        engine="pyarrow",
        dtype_backend="pyarrow",
    )

    required_cols = ["fileName", "caption"]
    missing = [col for col in required_cols if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    results = []

    for idx, row in df.iterrows():
        image_path = IMAGE_ROOT / row["fileName"]
        caption = row["caption"]

        print(f"\nRow {idx}")
        print("Image:", image_path)
        print("Caption:", caption)

        for model_name in MODELS:
            try:
                prediction = ask_gpt(
                    model_name=model_name,
                    image_path=str(image_path),
                    caption=caption,
                )

                result_row = {
                    "index": idx,
                    "model": model_name,
                    "fileName": row["fileName"],
                    "caption": caption,
                    "prediction": prediction,
                    "status": "ok",
                }

                print(prediction)
                print(f"[{model_name}] OK")

            except Exception as e:
                result_row = {
                    "index": idx,
                    "model": model_name,
                    "fileName": row["fileName"],
                    "caption": caption,
                    "prediction": None,
                    "status": f"error: {e}",
                }

                print(f"[{model_name}] ERROR: {e}")

            results.append(result_row)
            time.sleep(0.5)

    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_CSV, index=False)

    print("\nDone.")
    print(results_df.head())


if __name__ == "__main__":
    main()